# Tuned parametric PINN–FEM comparison

Compare the tuned parametric PINN with the FEM reference for $a_1=0.8$ and $a_2=0.6$. The notebook validates the saved parameter values and snapshot coordinates before plotting.

In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

from params import a1 as fem_a1, a2 as fem_a2, hole_r, hole_x, hole_y


EXPECTED_A1 = 0.8
EXPECTED_A2 = 0.6
FEM_FILE = Path("comparison_results/fem_comparison_snapshots.npz")
PINN_FILE = Path("parametric_training_tuned/pinn_parametric_tuned_nominal_snapshots.npz")
FIGURE_FILE = Path("comparison_snapshots_parametric_tuned_a1_08_a2_06.png")

In [ ]:
def require_file(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required snapshot file not found: {path}. "
            "Run the corresponding FEM or tuned parametric PINN training first."
        )


def require_keys(path: Path, available: set[str], required: set[str]) -> None:
    missing = required - available
    if missing:
        raise KeyError(f"{path} is missing required arrays: {sorted(missing)}")


require_file(FEM_FILE)
require_file(PINN_FILE)

with np.load(FEM_FILE) as data:
    require_keys(FEM_FILE, set(data.files), {"points", "times", "u"})
    fem_points = data["points"].copy()
    fem_times = data["times"].copy()
    u_fem = data["u"].copy()

with np.load(PINN_FILE) as data:
    require_keys(PINN_FILE, set(data.files), {"points", "times", "u", "a1", "a2"})
    pinn_points = data["points"].copy()
    pinn_times = data["times"].copy()
    u_pinn = data["u"].copy()
    pinn_a1 = float(np.asarray(data["a1"]).item())
    pinn_a2 = float(np.asarray(data["a2"]).item())

if not np.isclose(pinn_a1, EXPECTED_A1) or not np.isclose(pinn_a2, EXPECTED_A2):
    raise ValueError(
        f"Expected PINN snapshots for (a1, a2)=({EXPECTED_A1}, {EXPECTED_A2}), "
        f"but the archive contains ({pinn_a1}, {pinn_a2})"
    )
if not np.isclose(fem_a1, EXPECTED_A1) or not np.isclose(fem_a2, EXPECTED_A2):
    raise ValueError(
        f"params.py currently defines FEM diffusivities ({fem_a1}, {fem_a2}); "
        f"expected ({EXPECTED_A1}, {EXPECTED_A2})"
    )

print(f"Comparing FEM and tuned parametric PINN at a1={pinn_a1:g}, a2={pinn_a2:g}")

In [ ]:
if fem_points.ndim != 2 or fem_points.shape[1] != 2:
    raise ValueError(f"Expected FEM points with shape (n, 2), got {fem_points.shape}")
if pinn_points.shape != fem_points.shape:
    raise ValueError(
        f"Point-shape mismatch: FEM {fem_points.shape}, PINN {pinn_points.shape}"
    )
if fem_times.ndim != 1 or pinn_times.shape != fem_times.shape:
    raise ValueError(
        f"Time-shape mismatch: FEM {fem_times.shape}, PINN {pinn_times.shape}"
    )
expected_solution_shape = (len(fem_times), len(fem_points))
if u_fem.shape != expected_solution_shape or u_pinn.shape != expected_solution_shape:
    raise ValueError(
        f"Expected solution shape {expected_solution_shape}; "
        f"got FEM {u_fem.shape} and PINN {u_pinn.shape}"
    )
if not np.allclose(fem_points, pinn_points):
    raise ValueError("FEM and PINN snapshot coordinates do not match")
if not np.allclose(fem_times, pinn_times):
    raise ValueError("FEM and PINN snapshot times do not match")

arrays = {
    "FEM points": fem_points,
    "PINN points": pinn_points,
    "FEM times": fem_times,
    "PINN times": pinn_times,
    "FEM solution": u_fem,
    "PINN solution": u_pinn,
}
for name, values in arrays.items():
    if not np.isfinite(values).all():
        raise ValueError(f"{name} contains non-finite values")

points = fem_points
times = fem_times
x = np.unique(points[:, 0])
y = np.unique(points[:, 1])
x_grid, y_grid = np.meshgrid(x, y, indexing="xy")
x_indices = np.searchsorted(x, points[:, 0])
y_indices = np.searchsorted(y, points[:, 1])

if not (
    np.allclose(x[x_indices], points[:, 0])
    and np.allclose(y[y_indices], points[:, 1])
):
    raise ValueError("Comparison points cannot be mapped to the Cartesian grid")
flat_indices = np.ravel_multi_index((y_indices, x_indices), x_grid.shape)
if len(np.unique(flat_indices)) != len(points):
    raise ValueError("Comparison points contain duplicate grid locations")

print(
    f"Validated {len(points):,} points and {len(times)} snapshot times; "
    f"solution shape={u_fem.shape}"
)

In [ ]:
rows = len(times)
fig, axes = plt.subplots(
    rows,
    3,
    figsize=(14, 4 * rows),
    squeeze=False,
)
fig.suptitle(
    f"Tuned parametric PINN vs FEM ($a_1={pinn_a1:g}$, $a_2={pinn_a2:g}$)",
    fontsize=16,
)

for i, time in enumerate(times):
    pinn_field = np.full(x_grid.shape, np.nan)
    fem_field = np.full(x_grid.shape, np.nan)
    pinn_field[y_indices, x_indices] = u_pinn[i]
    fem_field[y_indices, x_indices] = u_fem[i]
    error_field = np.abs(pinn_field - fem_field)

    difference_norm = np.linalg.norm(u_pinn[i] - u_fem[i])
    fem_norm = np.linalg.norm(u_fem[i])
    if fem_norm > 0.0:
        relative_l2_percent = 100.0 * difference_norm / fem_norm
    else:
        relative_l2_percent = 0.0 if difference_norm == 0.0 else np.inf

    combined = np.concatenate((u_pinn[i], u_fem[i]))
    value_min = combined.min()
    value_max = combined.max()
    if value_min == value_max:
        padding = max(abs(value_min), 1.0) * 1.0e-12
        value_min -= padding
        value_max += padding

    pinn_image = axes[i, 0].pcolormesh(
        x_grid,
        y_grid,
        pinn_field,
        shading="auto",
        cmap="viridis",
        vmin=value_min,
        vmax=value_max,
    )
    fem_image = axes[i, 1].pcolormesh(
        x_grid,
        y_grid,
        fem_field,
        shading="auto",
        cmap="viridis",
        vmin=value_min,
        vmax=value_max,
    )
    error_image = axes[i, 2].pcolormesh(
        x_grid,
        y_grid,
        error_field,
        shading="auto",
        cmap="magma",
        vmin=0.0,
    )

    axes[i, 0].set_title(f"Parametric PINN: t = {time:g}")
    axes[i, 1].set_title(f"FEM: t = {time:g}")
    axes[i, 2].set_title(
        f"Absolute error (relative $L^2$ = {relative_l2_percent:.2f}%)"
    )

    for axis in axes[i]:
        axis.add_patch(
            plt.Circle((0.0, 0.0), 1.0, fill=False, color="black", linewidth=0.8)
        )
        axis.add_patch(
            plt.Circle(
                (hole_x, hole_y),
                hole_r,
                fill=False,
                color="black",
                linewidth=0.8,
            )
        )
        axis.set(
            aspect="equal",
            xlabel="x",
            ylabel="y",
            xlim=(-1.05, 1.05),
            ylim=(-1.05, 1.05),
        )

    fig.colorbar(pinn_image, ax=axes[i, 0], label="u")
    fig.colorbar(fem_image, ax=axes[i, 1], label="u")
    fig.colorbar(
        error_image,
        ax=axes[i, 2],
        label=r"$|u_{\mathrm{PINN}}-u_{\mathrm{FEM}}|$",
    )

fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.985))
fig.savefig(FIGURE_FILE, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved comparison figure to {FIGURE_FILE}")